In [4]:
import os

def count_gemini_files(directory):
    count = 0
    for filename in os.listdir(directory):
        if filename.endswith('.md') and '3-5-sonnet' in filename.lower():
            count += 1
    return count

# Example usage
# directory_path = '/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/blank_spacing/images/Acknowledgement'
# directory_path = '/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/blank_spacing/images/Subdermatoglyphic'
directory_paths = ['/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/templated_spaced/images_circled/tHyUiKaRbNqWeOpXcZvM',
                   '/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/templated_spaced/images_circled/Subdermatoglyphic',
                   '/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/templated_spaced/images_circled/Acknowledgement']   
for directory_path in directory_paths:
    result = count_gemini_files(directory_path)
    print(f"Number of .md files with 'gemini' in the filename: {result}")

Number of .md files with 'gemini' in the filename: 480
Number of .md files with 'gemini' in the filename: 408
Number of .md files with 'gemini' in the filename: 360


In [80]:
import os
import shutil


def remove_non_gemini_files(directory):
    # Ensure the directory exists
    if not os.path.exists(directory):
        print(f"The directory {directory} does not exist.")
        return

    # Walk through the directory and its subdirectories
    for root, dirs, files in os.walk(directory, topdown=False):
        for filename in files:
            file_path = os.path.join(root, filename)
            
            # Check if the filename doesn't contain "gemini"
            if "gemini" not in filename.lower():
                try:
                    os.remove(file_path)
                    print(f"Removed: {file_path}")
                except Exception as e:
                    print(f"Error removing {file_path}: {e}")
        
        # Remove empty directories
        for dir_name in dirs:
            dir_path = os.path.join(root, dir_name)
            try:
                os.rmdir(dir_path)
                print(f"Removed empty directory: {dir_path}")
            except OSError:
                # Directory not empty, skip it
                pass

# Usage
directory_path = "/Users/log/Desktop/gemini_final/highlight"
remove_non_gemini_files(directory_path)

Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i10_g_s3_t0.3_fOS.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i8_t_s2_t0.5_fOS.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i14_h_s0_t0.5_fH.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i8_t_s1_t0.4_fH.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i3_d_s0_t0.4_fH.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i10_g_s3_t0.3_fH.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i5_r_s1_t0.5_fH.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i14_h_s2_t0.3_fOS.png
Removed: /Users/log/Desktop/gemini_final/highlight/Subdermatoglyphic/Subdermatoglyphic_i7_a_s2_t0.5_fH.png
Removed: /Users/log/Desktop/ge

# Export

In [2]:
def extract_marked_text(text):
    start = text.find('{')
    end = text.find('}')
    
    if start != -1 and end != -1 and start < end:
        answer = text[start + 1 : end]
        return answer.lower()
    else:
        return "No Char"

# test = """The letter being circled in the image is:

# {}
# The red circle is surrounding the lowercase letter 'i' at the beginning of the string of letters shown."""
# print(extract_marked_text(test))

In [3]:
import pandas as pd
import os

# Define the words
WORDs = [
    "Acknowledgement",
    "Subdermatoglyphic",
    "tHyUiKaRbNqWeOpXcZvM",
]

# Initialize an empty list to store DataFrames
all_data_frames = []

# Loop through each word
for WORD in WORDs:
    # "/Users/log/Desktop/gemini_final/highlight/{WORD}/configurations_combined.json"
    # gt_data = pd.read_json(f"/Users/log/Desktop/gemini_final/highlight/{WORD}/configurations_combined.json")
    # gt_data = pd.read_json(f"/Users/log/Github/BlindLLMs/revisions/CircledWordRevisions/templated_spaced/images/tHyUiKaRbNqWeOpXcZvM/configurations_combined.json")
    gt_data = pd.read_json(f"/Users/log/Github/VideoFilters/images/{WORD}/configurations_combined.json")


    # remplace ./images/ with ./images_second_prompt/
    # gt_data["image_path"] = gt_data["image_path"].apply(
    #     lambda x: x.replace("./images/", "./Users/log/Github/VideoFilters/images/")
    # )

    # Generate the output file paths and read the content
    gt_data["model-output-file"] = gt_data["image_path"].apply(
        lambda x: "./" + x.replace(".png", "") + "-gemini-output.md"
    )
    gt_data["model-output-raw"] = gt_data["model-output-file"].apply(
        lambda x: open(x, "r").read() if os.path.exists(x) else None
    )

    # Drop rows with missing gemini output
    gt_data = gt_data.dropna(subset=["model-output-raw"])

    gt_data["predicted"] = gt_data["model-output-raw"].apply(extract_marked_text)

    print(gt_data["predicted"].value_counts())

    # Prepare the cleaned data
    cleaned_data = gt_data.copy()
    
    # cleaned_data["gt"] = cleaned_data.apply(
    #     lambda row: row["word"][row["circle_index"]].lower(), axis=1
    # )
    
    cleaned_data["gt"] = cleaned_data.apply(
        lambda row: row["circled_letter"].lower(), axis=1
    )   
    cleaned_data["is_prediction_correct"] = (
        cleaned_data["gt"] == cleaned_data["predicted"]
    )
    cleaned_data["word_label"] = WORD  # Add a column to identify the word

    # Append to the list

    all_data_frames.append(cleaned_data)

# Concatenate all DataFrames into one
final_data_frame = pd.concat(all_data_frames, ignore_index=True)

predicted
e    81
n    47
w    34
o    25
g    25
a    24
k    24
m    24
t    24
d    23
c    22
l     5
ⓒ     2
Name: count, dtype: int64
predicted
o    36
g    34
y    31
e    29
d    28
s    27
h    25
m    24
p    24
c    24
i    22
u    21
b    19
r    18
t    18
@    15
a     9
0     4
Name: count, dtype: int64
predicted
b    31
p    28
e    27
z    26
c    26
y    25
m    25
u    25
i    25
n    25
t    24
a    23
x    23
w    21
v    20
o    20
h    19
q    17
r    16
k    14
@    11
g     7
j     1
l     1
Name: count, dtype: int64


In [4]:
final_data_frame["Model"] = ["Gemini-1.5-Pro"] * len(final_data_frame)

In [6]:
final_data_frame.to_pickle("gemini-1.5-pro-highlighted.pkl")

In [6]:
final_data_frame

,word,font_path,circle_index,circled_letter,thickness,scale_factor,padding,canvas_width,canvas_height,final_width,final_height,num_spaces,image_path,model-output-file,model-output-raw,predicted,gt,is_prediction_correct,word_label,Model
0,Acknowledgement,fonts/helvetica.ttf,0,A,0.3,1.4,25,10,2,1250,1250,0,./images/Acknowledgement/Acknowledgement_i0_A_...,././images/Acknowledgement/Acknowledgement_i0_...,{A},a,a,True,Acknowledgement,Gemini-1.5-Pro
1,Acknowledgement,fonts/helvetica.ttf,0,A,0.4,1.4,25,10,2,1250,1250,0,./images/Acknowledgement/Acknowledgement_i0_A_...,././images/Acknowledgement/Acknowledgement_i0_...,{A},a,a,True,Acknowledgement,Gemini-1.5-Pro
2,Acknowledgement,fonts/helvetica.ttf,0,A,0.5,1.4,25,10,2,1250,1250,0,./images/Acknowledgement/Acknowledgement_i0_A_...,././images/Acknowledgement/Acknowledgement_i0_...,{A},a,a,True,Acknowledgement,Gemini-1.5-Pro
3,Acknowledgement,fonts/OpenSans-Regular.ttf,0,A,0.3,1.4,25,10,2,1250,1250,0,./images/Acknowledgement/Acknowledgement_i0_A_...,././images/Acknowledgement/Acknowledgement_i0_...,{A},a,a,True,Acknowledgement,Gemini-1.5-Pro
4,Acknowledgement,fonts/OpenSans-Regular.ttf,0,A,0.4,1.4,25,10,2,1250,1250,0,./images/Acknowledgement/Acknowledgement_i0_A_...,././images/Acknowledgement/Acknowledgement_i0_...,{A} \n,a,a,True,Acknowledgement,Gemini-1.5-Pro
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1243,tHyUiKaRbNqWeOpXcZvM,fonts/helvetica.ttf,19,M,0.4,1.4,25,10,2,1250,1250,3,./images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeOpX...,././images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeO...,{M},m,m,True,tHyUiKaRbNqWeOpXcZvM,Gemini-1.5-Pro
1244,tHyUiKaRbNqWeOpXcZvM,fonts/helvetica.ttf,19,M,0.5,1.4,25,10,2,1250,1250,3,./images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeOpX...,././images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeO...,{M} \n,m,m,True,tHyUiKaRbNqWeOpXcZvM,Gemini-1.5-Pro
1245,tHyUiKaRbNqWeOpXcZvM,fonts/OpenSans-Regular.ttf,19,M,0.3,1.4,25,10,2,1250,1250,3,./images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeOpX...,././images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeO...,{M},m,m,True,tHyUiKaRbNqWeOpXcZvM,Gemini-1.5-Pro
1246,tHyUiKaRbNqWeOpXcZvM,fonts/OpenSans-Regular.ttf,19,M,0.4,1.4,25,10,2,1250,1250,3,./images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeOpX...,././images/tHyUiKaRbNqWeOpXcZvM/tHyUiKaRbNqWeO...,{M},m,m,True,tHyUiKaRbNqWeOpXcZvM,Gemini-1.5-Pro
